In [30]:
import numpy as np
import c832 as c
import CircuitBuilder
from tqdm import tqdm

In [83]:
defaultzschedule = np.asarray([
    [1, 2, 3, 4, 0, 0, 0, 0],
    [0, 0, 0, 0, 1, 2, 3, 4],
    [2, 3, 0, 0, 4, 1, 0, 0],
    [3, 0, 4, 0, 2, 0, 1, 0]
])
defaultxschedule = np.asarray([
    [1, 2, 3, 4, 5, 6, 7, 8]
])

def windowFiller(schedule, nwindow):
    thiswindow = []
    for row in schedule:
        if nwindow in row:
            thiswindow.append(np.where(row == nwindow)[0][0])
        else:
            thiswindow.append(None)
    return thiswindow

def c832scheduleableinjectable(er = 0, nrounds = 1, zschedule = defaultzschedule, xschedule = defaultxschedule, errorschedule = None):
    zschedule, xschedule = np.asarray(zschedule), np.asarray(xschedule)
    #cb = CircuitBuilder.CircuitBuilder(er, er, er, er, er, 0)
    cb = CircuitBuilder.CircuitBuilder(er, er, er, er, er, er)
    cb.al('R 0 1 2 3 4 5 6 7')
    cb.al('H 0 1 2 3 4 5 6 7')
    nancillas = len(zschedule) + len(xschedule)
    ancillastring = 'R ' + ' '.join([f'{i}' for i in range(8, 8 + nancillas)])
    cb.al(ancillastring)
    cb.set_ancilla_qubits(qubitslistz = [8 + i for i in range(len(zschedule))], qubitslistx = [8 + len(zschedule) + i for i in range(len(xschedule))])
    #print(cb.ancilla_qubitsz, cb.ancilla_qubitsx)
    # Warmup round
    for windowz in range(1, np.max(zschedule) + 1):
        #cb.time_tick()
        #print(windowFiller(zschedule, windowz))
        cb.couple_z_check(windowFiller(zschedule, windowz), False)
    cb.al('MR(0) ' + ' '.join([f'{i}' for i in cb.ancilla_qubitsz]))
    cb.al('H ' + ' '.join([f'{i}' for i in cb.ancilla_qubitsx]))
    for windowx in range(1, np.max(xschedule) + 1):
        #cb.time_tick()
        cb.couple_x_check(windowFiller(xschedule, windowx), False)
    cb.al('H ' + ' '.join([f'{i}' for i in cb.ancilla_qubitsx]))
    cb.al('MR(0) ' + ' '.join([f'{i}' for i in cb.ancilla_qubitsx]))
    cb.al('R ' + ' '.join([f'{i}' for i in cb.ancilla_qubitsz]))
    # Noise rounds
    cb.begin_loop(nrounds)
    timewindow = 0
    for windowz in range(1, np.max(zschedule) + 1):
        cb.time_tick()
        if errorschedule is not None:
            for errornum in range(len(errorschedule[timewindow])):
                if errorschedule[timewindow][errornum] != 'I':
                    cb.inject_error([errornum], errorschedule[timewindow][errornum])
        timewindow += 1
        cb.couple_z_check(windowFiller(zschedule, windowz), True)
    cb.al('MR(0) ' + ' '.join([f'{i}' for i in cb.ancilla_qubitsz]))
    cb.al('R ' + ' '.join([f'{i}' for i in cb.ancilla_qubitsx]))
    cb.al('H ' + ' '.join([f'{i}' for i in cb.ancilla_qubitsx]))
    for windowx in range(1, np.max(xschedule) + 1):
        cb.time_tick()
        if errorschedule is not None:
            for errornum in range(len(errorschedule[timewindow])):
                if errorschedule[timewindow][errornum] != 'I':
                    cb.inject_error([errornum], errorschedule[timewindow][errornum])
        timewindow += 1
        cb.couple_x_check(windowFiller(xschedule, windowx), True)
    cb.add_detectors([[-1 - i, -1 - i - nancillas] for i in range(len(cb.ancilla_qubitsz))])
    cb.al('H ' + ' '.join([f'{i}' for i in cb.ancilla_qubitsx]))
    cb.al('MR(0) ' + ' '.join([f'{i}' for i in cb.ancilla_qubitsx]))
    cb.add_detectors([[-1 - i, -1 - i - nancillas] for i in range(len(cb.ancilla_qubitsx))])
    cb.al('R ' + ' '.join([f'{i}' for i in cb.ancilla_qubitsz]))
    cb.end_loop()
    # Final measurement of all physical qubits
    cb.al('H 0 1 2 3 4 5 6 7')
    cb.al(f'MR({0}) 0 1 2 3 4 5 6 7')
    return cb

def detector_builder(mseq, syndromelength = 5, nphysicalqubits = 8, nrounds = 1, mode = 'binary'):
    if len(mseq) != syndromelength * (nrounds + 1) + nphysicalqubits:
        raise ValueError(f"Measurement sequence length must be equal to syndromelength * nrounds + nphysicalqubits")
    syndromes, finalmeasurement = mseq[:-nphysicalqubits], mseq[-nphysicalqubits:]
    if mode == 'string':
        det = []
        for i in range(nrounds):
            det.append(''.join([str(0 if x == y else 1) for x, y in zip(syndromes[i*syndromelength:(i+1)*syndromelength], syndromes[(i+1)*syndromelength:(i+2)*syndromelength])]))
    elif mode == 'binary':
        det = np.zeros(nrounds * syndromelength, dtype = np.bool)
        for i in range(nrounds):
            det[i*syndromelength:(i+1)*syndromelength] = np.array([0 if x == y else 1 for x, y in zip(syndromes[i*syndromelength:(i+1)*syndromelength], syndromes[(i+1)*syndromelength:(i+2)*syndromelength])])
    else:
        raise ValueError(f"Invalid mode: {mode}")

    return det, finalmeasurement

def check_detector_fired(det):
    for detector in det:
        if '1' in detector:
            return True
    return False

def measure_readout(readout, logical_operator):
    parity = 0
    for i in range(len(readout)):
        if readout[i] == '1' and logical_operator[i]:
            parity += 1
    return parity % 2


In [84]:
zschedule = [[1, 4, 2, 3, 0, 0, 0, 0],
 [0, 0, 0, 0, 2, 3, 1, 4],
 [3, 2, 0, 0, 4, 1, 0, 0],
 [2, 0, 3, 0, 1, 0, 4, 0]]
xschedule = [[1, 0, 4, 0, 0, 3, 0, 2],
 [2, 0, 3, 0, 4, 0, 1, 0],
 [0, 1, 0, 4, 3, 0, 2, 0]]
errorschedule = [
    ['I', 'I', 'I', 'I', 'I', 'I', 'I', 'I'],
    ['I', 'I', 'I', 'I', 'I', 'I', 'I', 'I'],
    ['I', 'I', 'I', 'I', 'I', 'I', 'I', 'I'],
    ['I', 'I', 'I', 'I', 'I', 'I', 'I', 'I'],
    ['I', 'I', 'I', 'I', 'I', 'I', 'I', 'I'],
    ['I', 'I', 'I', 'I', 'I', 'I', 'I', 'I'],
    ['I', 'I', 'I', 'I', 'I', 'I', 'I', 'I'],
    ['I', 'I', 'I', 'I', 'I', 'I', 'I', 'I']
]
cb = c832scheduleableinjectable(er = 0, nrounds = 1, zschedule = zschedule, xschedule = xschedule, errorschedule = errorschedule)

In [38]:
cb.get_circuit().diagram()

/-------------------------------------------------------------------------------------\ /--------------------\ /--------------------\ /--------------------\ /------------------------------------\ /------------------\ /------------------\ /------------------\ /----------------------------------------------------------------------------------------------------\
 q0: -R-H-@-------------@-----@------------------------X-------X------------------------------DEPOLARIZE1(0)-@-------DEPOLARIZE1(0)-------@-DEPOLARIZE1(0)-----@---DEPOLARIZE1(0)-------------------------DEPOLARIZE1(0)-X-----DEPOLARIZE1(0)---X---DEPOLARIZE1(0)-X-----DEPOLARIZE1(0)------------------------------------------------H--------------------------MR(0):rec[14]-
          |             |     |                        |       |                                             |                            |                    |                                                         |                      |
 q1: -R-H-|-----------@-|-----|---@--------------------|---X---|------------------------------DEPOLARIZE1(0)-|-------DEPOLARIZE1(0)-----@-|-DEPOLARIZE1(0)-----|---DEPOLARIZE1(0)-@-----------------------DEPOLARIZE1(0)-|---X-DEPOLARIZE1(0)---|---DEPOLARIZE1(0)-------DEPOLARIZE1(0)------------------------------------------------H--------------------------MR(0):rec[15]-
          |           | |     |   |                    |   |   |                                             |                          | |                    |                  |                                      |   |                  |
 q2: -R-H-|-------@---|-|-----|-@-|--------------------|---|---|-----X---X--------------------DEPOLARIZE1(0)-|-------DEPOLARIZE1(0)-@---|-|-DEPOLARIZE1(0)-----|-@-DEPOLARIZE1(0)-|-----------------------DEPOLARIZE1(0)-|---|-DEPOLARIZE1(0)---|---DEPOLARIZE1(0)---X---DEPOLARIZE1(0)-X----------------------------------------------H--------------------------MR(0):rec[16]-
          |       |   | |     | | |                    |   |   |     |   |                                   |                      |   | |                    | |                |                                      |   |                  |                    |                  |
 q3: -R-H-|-------|---|-|-@---|-|-|--------------------|---|---|-----|---|---X----------------DEPOLARIZE1(0)-|-------DEPOLARIZE1(0)-|---|-|-DEPOLARIZE1(0)-@---|-|-DEPOLARIZE1(0)-|-----------------------DEPOLARIZE1(0)-|---|-DEPOLARIZE1(0)---|---DEPOLARIZE1(0)---|---DEPOLARIZE1(0)-|---X------------------------------------------H--------------------------MR(0):rec[17]-
          |       |   | | |   | | |                    |   |   |     |   |   |                               |                      |   | |                |   | |                |                                      |   |                  |                    |                  |   |
 q4: -R-H-|-----@-|-@-|-|-|---|-|-|---@----------------|---|---|-----|-X-|-X-|----------------DEPOLARIZE1(0)-|-----@-DEPOLARIZE1(0)-|-@-|-|-DEPOLARIZE1(0)-|---|-|-DEPOLARIZE1(0)-|---@-------------------DEPOLARIZE1(0)-|---|-DEPOLARIZE1(0)---|---DEPOLARIZE1(0)---|-X-DEPOLARIZE1(0)-|-X-|------------------------------------------H--------------------------MR(0):rec[18]-
          |     | | | | | |   | | |   |                |   |   |     | | | | |                               |     |                | | | |                |   | |                |   |                                  |   |                  |                    | |                | | |
 q5: -R-H-|---@-|-|-|-|-|-|-@-|-|-|---|----------------|---|---|---X-|-|-|-|-|----------------DEPOLARIZE1(0)-|---@-|-DEPOLARIZE1(0)-|-|-|-|-DEPOLARIZE1(0)-|-@-|-|-DEPOLARIZE1(0)-|---|-------------------DEPOLARIZE1(0)-|---|-DEPOLARIZE1(0)---|---DEPOLARIZE1(0)-X-|-|-DEPOLARIZE1(0)-|-|-|------------------------------------------H--------------------------MR(0):rec[19]-
          |   | | | | | | | | | | |   |                |   |   |   | | | | | |        

In [41]:
nshots = 100000
sample = cb.sample(nshots)
count = 0
for row in tqdm(sample):
    det = detector_builder(row[0], 7, 8, 1, mode = 'string')[0]
    if '1' not in det[0]:
        count += row[1]
print(count / nshots)

100%|██████████| 256/256 [00:00<00:00, 639893.82it/s]

1.0


In [85]:
zschedule1 = [[3, 4, 1, 2, 0, 0, 0, 0],
 [0, 0, 0, 0, 4, 3, 2, 1],
 [1, 3, 0, 0, 2, 4, 0, 0],
 [4, 0, 2, 0, 1, 0, 3, 0]]
xschedule1 = [[4, 0, 1, 0, 0, 3, 0, 2],
 [3, 0, 4, 0, 1, 0, 2, 0],
 [0, 1, 0, 2, 4, 0, 3, 0]]
zschedule2 = [[1, 4, 2, 3, 0, 0, 0, 0],
 [0, 0, 0, 0, 2, 3, 1, 4],
 [3, 2, 0, 0, 4, 1, 0, 0],
 [2, 0, 3, 0, 1, 0, 4, 0]]
xschedule2 = [[1, 0, 4, 0, 0, 3, 0, 2],
 [2, 0, 3, 0, 4, 0, 1, 0],
 [0, 1, 0, 4, 3, 0, 2, 0]]
emptyerrorschedule = [
    ['I', 'I', 'I', 'I', 'I', 'I', 'I', 'I'],
    ['I', 'I', 'I', 'I', 'I', 'I', 'I', 'I'],
    ['I', 'I', 'I', 'I', 'I', 'I', 'I', 'I'],
    ['I', 'I', 'I', 'I', 'I', 'I', 'I', 'I'],
    ['I', 'I', 'I', 'I', 'I', 'I', 'I', 'I'],
    ['I', 'I', 'I', 'I', 'I', 'I', 'I', 'I'],
    ['I', 'I', 'I', 'I', 'I', 'I', 'I', 'I'],
    ['I', 'I', 'I', 'I', 'I', 'I', 'I', 'I'],
]
#zschedule, xschedule = zschedule1, xschedule1
zschedule, xschedule = zschedule2, xschedule2
errorprob = 0
for errortype in ['X', 'Z', 'Y']:
    for rowcol in tqdm(range(8 * 8)):
        row, col = rowcol // 8, rowcol % 8
        errorschedule = emptyerrorschedule.copy()
        errorschedule[row][col] = errortype
        cb = c832scheduleableinjectable(er = 0, nrounds = 1, zschedule = zschedule, xschedule = xschedule, errorschedule = errorschedule)
        nshots = 10000
        sample = cb.sample(nshots)
        count = 0
        for row in sample:
            det = detector_builder(row[0], 7, 8, 1, mode = 'string')[0]
            if '1' not in det[0]:
                count += row[1]
        errorprob += count / nshots
print(errorprob)

100%|██████████| 64/64 [00:00<00:00, 74.87it/s]

82.0


In [89]:
#zschedule, xschedule = zschedule1, xschedule1
zschedule, xschedule = zschedule2, xschedule2
cb = c832scheduleableinjectable(er = 0.001, nrounds = 1, zschedule = zschedule, xschedule = xschedule)
sample = cb.sample(1000000)
logop = np.array([1, 1, 0, 0, 1, 1, 0, 0])
s, sandc = 0, 0
for row in tqdm(sample):
    det, finalmeasurement = detector_builder(row[0], 7, 8, 1, mode = 'string')
    correctreadout = measure_readout(finalmeasurement, logop) == 0
    survives = not check_detector_fired(det)
    if survives:
        s += row[1]
    if survives and correctreadout:
        sandc += row[1]
print(1 - (sandc / s), s - sandc, s)

100%|██████████| 18776/18776 [00:00<00:00, 508439.87it/s]

0.006243230215835172 5712 914911


In [82]:
emptyerrorschedule = [
    ['I', 'I', 'I', 'I', 'I', 'I', 'I', 'I', 'I', 'I', 'I', 'I', 'I', 'I', 'I'],
    ['I', 'I', 'I', 'I', 'I', 'I', 'I', 'I', 'I', 'I', 'I', 'I', 'I', 'I', 'I'],
    ['I', 'I', 'I', 'I', 'I', 'I', 'I', 'I', 'I', 'I', 'I', 'I', 'I', 'I', 'I'],
    ['I', 'I', 'I', 'I', 'I', 'I', 'I', 'I', 'I', 'I', 'I', 'I', 'I', 'I', 'I'],
    ['I', 'I', 'I', 'I', 'I', 'I', 'I', 'I', 'I', 'I', 'I', 'I', 'I', 'I', 'I'],
    ['I', 'I', 'I', 'I', 'I', 'I', 'I', 'I', 'I', 'I', 'I', 'I', 'I', 'I', 'I'],
    ['I', 'I', 'I', 'I', 'I', 'I', 'I', 'I', 'I', 'I', 'I', 'I', 'I', 'I', 'I'],
    ['I', 'I', 'I', 'I', 'I', 'I', 'I', 'I', 'I', 'I', 'I', 'I', 'I', 'I', 'I'],
]
#zschedule, xschedule = zschedule1, xschedule1
zschedule, xschedule = zschedule2, xschedule2
errorprob = 0
for errortype in ['X', 'Z', 'Y']:
    for rowcol in tqdm(range(15 * 8)):
        row, col = rowcol // 15, rowcol % 8
        errorschedule = emptyerrorschedule.copy()
        errorschedule[row][col] = errortype
        cb = c832scheduleableinjectable(er = 0, nrounds = 1, zschedule = zschedule, xschedule = xschedule, errorschedule = errorschedule)
        nshots = 1000
        sample = cb.sample(nshots)
        count = 0
        for row in sample:
            det = detector_builder(row[0], 7, 8, 1, mode = 'string')[0]
            if '1' not in det[0]:
                count += row[1]
        errorprob += count / nshots
print(errorprob)

100%|██████████| 120/120 [00:00<00:00, 217.22it/s]

196.0
